In [1]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL_ID = "google/gemma-4-E2B-it"

# Load model
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, 
    dtype="auto", 
    device_map="auto"
)

/home/monster/Desktop/codes/conversation_agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 1951/1951 [00:02<00:00, 741.23it/s] 


In [12]:
import tempfile
import sounddevice as sd
import soundfile as sf
import torch
import numpy as np

# Match the sample rate expected by the model's audio processor.
sample_rate = getattr(processor, "feature_extractor", None)
sample_rate = getattr(sample_rate, "sampling_rate", 16000)

chunk_seconds = 5

prompt = (
    "Transcribe the following speech segment in its original language. "
    "Only output the transcription, with no newlines. "
    "When transcribing numbers, write digits: write 1.7, not one point seven, "
    "and write 3, not three."
)

def transcribe_chunk(audio):
    # Save the captured audio as a WAV file so the processor can load it.
    with tempfile.NamedTemporaryFile(suffix=".wav") as wav_file:
        sf.write(wav_file.name, audio, sample_rate)

        messages = [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "audio", "audio": wav_file.name},
            ],
        }]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
            add_generation_prompt=True,
        ).to(model.device)

        input_len = inputs["input_ids"].shape[-1]

        with torch.inference_mode():
            outputs = model.generate(**inputs, max_new_tokens=512)

        response = processor.decode(
            outputs[0][input_len:],
            skip_special_tokens=True,
        )
        return response.strip()

chunk_seconds = 5
threshold = 0.01  # Raise this if background noise triggers transcription

print(f"Listening in {chunk_seconds}-second chunks. Press Ctrl+C to stop.")

try:
    while True:
        audio = sd.rec(
            int(chunk_seconds * sample_rate),
            samplerate=sample_rate,
            channels=1,
            dtype="float32",
        )
        sd.wait()

        audio = audio.squeeze()
        level = float(np.sqrt(np.mean(audio ** 2)))
        print(f"Audio level: {level:.4f}")

        if level < threshold:
            continue

        text = transcribe_chunk(audio)
        if text:
            print(text, flush=True)

except KeyboardInterrupt:
    print("\nStopped listening.")

Listening in 5-second chunks. Press Ctrl+C to stop.
Audio level: 0.0000
Audio level: 0.0000
Audio level: 0.0000
Audio level: 0.0000
Audio level: 0.0323
What is transformer architecture explain in hindi
Audio level: 0.0419
Tell me about TransFormer architecture and explain in detail
Audio level: 0.0394
Hello testing testing
Audio level: 0.0315
Hello test and testing.
Audio level: 0.0158
This is a test.

Stopped listening.


In [10]:
import sounddevice as sd
print(sd.query_devices())
print("Default input:", sd.default.device)

   0 ThinkPad Hybrid USB-C with USB-: Audio (hw:0,0), ALSA (2 in, 6 out)
   1 Jabra Link 380: USB Audio (hw:1,0), ALSA (1 in, 2 out)
   2 HDA NVidia: DELL U3415W (hw:2,3), ALSA (0 in, 2 out)
   3 HDA NVidia: LC49G95T (hw:2,7), ALSA (0 in, 2 out)
   4 HDA NVidia: HDMI 2 (hw:2,8), ALSA (0 in, 8 out)
   5 HDA NVidia: HDMI 3 (hw:2,9), ALSA (0 in, 8 out)
   6 Microsoft® LifeCam HD-3000: USB Audio (hw:3,0), ALSA (1 in, 0 out)
   7 sof-hda-dsp: - (hw:4,0), ALSA (2 in, 2 out)
   8 sof-hda-dsp: P24h-10 (hw:4,3), ALSA (0 in, 2 out)
   9 sof-hda-dsp: HDMI 2 (hw:4,4), ALSA (0 in, 2 out)
  10 sof-hda-dsp: HDMI 3 (hw:4,5), ALSA (0 in, 2 out)
  11 sof-hda-dsp: - (hw:4,6), ALSA (2 in, 0 out)
  12 sof-hda-dsp: - (hw:4,7), ALSA (2 in, 0 out)
  13 sof-hda-dsp: - (hw:4,31), ALSA (0 in, 2 out)
  14 sysdefault, ALSA (128 in, 128 out)
  15 front, ALSA (0 in, 6 out)
  16 surround40, ALSA (0 in, 6 out)
  17 surround41, ALSA (0 in, 128 out)
  18 surround50, ALSA (0 in, 128 out)
  19 surround51, ALSA (0 in, 128 

In [2]:
# Prompt - add audio after text
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Transcribe the following speech segment in its original language. Follow these specific instructions for formatting the answer:\n* Only output the transcription, with no newlines.\n* When transcribing numbers, write the digits, i.e. write 1.7 and not one point seven, and write 3 instead of three."},
            {"type": "audio", "audio": "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/apps/sample-data/journal1.wav"},
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
processor.parse_response(response, prefix=inputs["input_ids"])


{'role': 'assistant',
 'content': 'a call of today feeling really fresh the morning right as beautiful and i enjoyed a nice cup of coffee'}